# Static Solve — Dragon Under Gravity

This notebook builds a static equilibrium solve for the dragon tet
volume mesh under gravity, using the display `dragon.obj` and the
provided `dragon-fixed.txt` fixed set.

The fixed file is authored against the tet volume mesh. The static
solve keeps those volume vertices near their rest positions with a
soft `VertexAttachment` term, while gravity and elasticity are
evaluated on the same tet volume mesh.


In [1]:
from pathlib import Path

import numpy as np
import pypgo as pgo
import pypgo.energy as pe
import pypgo.fem as pf
import pypgo.solver as ps
from pypgo.mesh import visualize as vis
from pypgo.mesh.volume import VolumeMesh, read_veg


## 1. Locate the Dragon Assets

The volume mesh is tetrahedral. The surface mesh is only for display;
the provided fixed vertex set is indexed on `dragon_big.veg`.


In [2]:
PACKAGE_ROOT = Path(pgo.__file__).resolve().parent.parent
ASSET_DIR = PACKAGE_ROOT / "examples" / "assets"
OUTPUT_DIR = PACKAGE_ROOT / "examples" / "outputs" / "static_solve_dragon_gravity"

DRAGON_TET = ASSET_DIR / "veg" / "tet" / "dragon_big.veg"
DRAGON_SURFACE = ASSET_DIR / "obj" / "dragon.obj"
DRAGON_FIXED = ASSET_DIR / "fixed" / "dragon-fixed.txt"

print("volume asset:", DRAGON_TET)
print("surface asset:", DRAGON_SURFACE)
print("fixed asset:", DRAGON_FIXED)
print("output dir:", OUTPUT_DIR)


volume asset: /Users/jinceyang/Desktop/codebase/libpgo/examples/assets/veg/tet/dragon_big.veg
surface asset: /Users/jinceyang/Desktop/codebase/libpgo/examples/assets/obj/dragon.obj
fixed asset: /Users/jinceyang/Desktop/codebase/libpgo/examples/assets/fixed/dragon-fixed.txt
output dir: /Users/jinceyang/Desktop/codebase/libpgo/examples/outputs/static_solve_dragon_gravity


## 2. Load the Tet Volume and Display Surface

`SurfaceEmbedding` creates the interpolation from volume displacement
DOFs to the high-resolution display surface.


In [3]:
veg = read_veg(str(DRAGON_TET))
volume = VolumeMesh.from_veg_file(veg)
tet_data = volume.mesh_data

rest_surface = pgo.mesh.read_obj(str(DRAGON_SURFACE))
surface_embedding = pgo.mesh.SurfaceEmbedding(rest_surface, volume)

print(volume)
print("tet geometry:", tet_data.num_vertices, "vertices,", tet_data.num_elements, "tets")
print("material:", volume.material)
print("tet bbox:", tet_data.bbox)
print("surface:", rest_surface.num_vertices, "vertices,", rest_surface.num_elements, "triangles")
print(
    "surface interpolation:",
    surface_embedding.interpolation_matrix.shape,
    "nnz:",
    surface_embedding.interpolation_matrix.nnz,
)

vis.plot_surface(
    rest_surface,
    titles=["dragon rest surface"],
    colors=["lightgray"],
    show_edges=False,
    window_size=(720, 520),
)


154164,38541,119937
VolumeMesh(type=MeshType.Tet, vertices=39979, elements=186736)
tet geometry: 39979 vertices, 186736 tets
material: ENuMaterial(name='defaultMaterial', density=1000.0, E=1000000.0, nu=0.45)
tet bbox: (array([-0.40752   , -0.408731  , -0.00071263]), array([0.30183956, 0.59291035, 0.446667  ]))
surface: 12847 vertices, 25694 triangles
surface interpolation: (38541, 119937) nnz: 154164


Widget(value='<iframe src="http://localhost:54947/index.html?ui=P_0x1763d46b0_0&reconnect=auto" class="pyvista…

## 3. Load the Fixed Volume Vertices

`dragon-fixed.txt` stores zero-based vertex IDs for `dragon_big.veg`.
These vertices are not hard constrained. They become a soft attachment
energy with coefficient `1e5`.


In [4]:
fixed_vertices = np.loadtxt(DRAGON_FIXED, dtype=np.int64).reshape(-1)
if fixed_vertices.size == 0:
    raise ValueError(f"{DRAGON_FIXED} does not contain any fixed IDs")
fixed_vertices = np.unique(fixed_vertices)
if fixed_vertices.min() < 0:
    raise ValueError("fixed vertex IDs must be non-negative")
if fixed_vertices.max() >= tet_data.num_vertices:
    raise ValueError(
        "fixed vertex IDs do not fit dragon_big.veg: "
        f"max={fixed_vertices.max()}, num_tet_vertices={tet_data.num_vertices}"
    )

fixed_positions = tet_data.vertices[fixed_vertices]
print("fixed volume vertices:", fixed_vertices.size)
print("fixed vertex ID range:", int(fixed_vertices.min()), int(fixed_vertices.max()))
print("first fixed vertices:", fixed_vertices[:30].tolist())
print("fixed volume bbox:", fixed_positions.min(axis=0), fixed_positions.max(axis=0))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fixed_points_obj = OUTPUT_DIR / "dragon_fixed_points.obj"
vis.write_points_obj(fixed_points_obj, fixed_positions)
print("wrote fixed points:", fixed_points_obj)

vis.plot_points_on_mesh(
    tet_data,
    fixed_positions,
    title="fixed volume vertices on dragon_big.veg",
    mesh_opacity=0.28,
    point_color="red",
    point_size=10,
    show_edges=False,
    window_size=(900, 650),
)


fixed volume vertices: 289
fixed vertex ID range: 3161 36645
first fixed vertices: [3161, 3209, 3221, 3300, 3305, 3320, 3362, 3369, 3440, 3455, 3463, 3464, 3472, 3482, 3489, 3511, 3514, 3520, 3522, 3534, 3539, 3576, 3577, 3599, 3624, 3626, 3630, 3637, 3638, 3648]
fixed volume bbox: [-0.182199    0.46456233  0.14830624] [-0.054505    0.59291035  0.220242  ]
wrote fixed points: /Users/jinceyang/Desktop/codebase/libpgo/examples/outputs/static_solve_dragon_gravity/dragon_fixed_points.obj


Widget(value='<iframe src="http://localhost:54947/index.html?ui=P_0x1765021e0_1&reconnect=auto" class="pyvista…

## 4. Build the Static Objective

The tet mesh uses `TetLinear()`. Gravity is converted to a generalized
force with the same formulation, then represented as a linear potential
energy `-f^T u`. The fixed volume vertices are kept near zero
displacement with `VertexAttachment(coeff=1e5)`.


In [5]:
sim_mesh = pgo.fem.SimulationMesh.create_volumetric(volume)
formulation = pf.TetLinear()
deformation = pf.deformation_energy(
    sim_mesh,
    elastic=pf.StableNeo(),
    elastic_field=pf.ElementwiseField(),
    plastic=pf.VolumetricPlasticity(dofs=0),
    plastic_field=pf.ElementwiseField(),
    formulation=formulation,
)

gravity_accel = np.array([0.0, -9.81, 0.0], dtype=np.float64)
gravity_force = formulation.body_force(volume, gravity_accel)
gravity_energy = pe.LinearEnergy(-gravity_force)

attachment_coeff = 1e5
volume_attachment = pe.VertexAttachment(
    sim_mesh=sim_mesh,
    vertex_indices=fixed_vertices,
    target_positions=np.zeros(3 * fixed_vertices.size, dtype=np.float64),
    coeff=attachment_coeff,
    is_displacement=True,
)

total_energy = pe.EnergySet([
    (deformation, 1.0),
    (gravity_energy, 1.0),
    (volume_attachment, 1.0),
])

print("mesh_type:", sim_mesh.mesh_type)
print("vertices:", sim_mesh.num_vertices)
print("elements:", sim_mesh.num_elements)
print("DOFs:", deformation.num_dofs)
print("state_kind:", deformation.state_kind)
print("gravity force norm:", float(np.linalg.norm(gravity_force)))
print("linear energy DOFs:", gravity_energy.num_dofs)
print("volume attachment coeff:", attachment_coeff)
print("volume attachment DOFs:", volume_attachment.num_dofs)


[2026-06-10 17:19:58.967] [info] [deformationModelManager.cpp:248] Initializing element models (manager path)...
[2026-06-10 17:19:59.443] [info] [deformationModelAssembler.cpp:227] Assembler parameter channels:0,0 local:0,0
mesh_type: tet
vertices: 39979
elements: 186736
DOFs: 119937
state_kind: displacement
gravity force norm: 3.406385395015793
linear energy DOFs: 119937
volume attachment coeff: 100000.0
volume attachment DOFs: 119937


## 5. Initialize the Static State

There are no hard fixed DOFs in this scene. The fixed volume vertices
are soft constraints in the objective with coefficient `1e5`.


In [6]:
x0 = np.zeros(deformation.num_dofs, dtype=np.float64)

print("initial DOFs:", x0.size)
print("volume attachment energy at x0:", volume_attachment.value(x0))


initial DOFs: 119937
volume attachment energy at x0: 0.0


## 6. Solve the Static Equilibrium

The static problem minimizes elastic energy, gravity potential, and
the volume soft attachment energy. `dragon_big.veg` is a large mesh,
so this solve can take substantially longer than the smaller example
notebooks.


In [ ]:
problem = ps.OptimizationProblem(objective=total_energy)

optimizer = ps.NewtonOptimizer(
    max_iterations=300,
    gradient_tolerance=1e-6,
    damping=True,
    line_search=ps.Simple(),
)
result = optimizer.solve(problem, x0)

print("status:", result.status.name)
print("converged:", result.converged)
print("iterations:", result.iterations)
print("final objective:", result.final_objective)
print("final gradient max norm:", result.final_gradient_max_norm)
print("max |u|:", float(np.max(np.abs(result.x))))
fixed_displacement = result.x.reshape((-1, 3))[fixed_vertices]
print("volume attachment energy after solve:", volume_attachment.value(result.x))
print("max fixed volume |u|:", float(np.max(np.abs(fixed_displacement))))


## 7. Export and Visualize the Deformed Dragon

The optimized state is a volume displacement vector. `SurfaceEmbedding`
maps it back onto `dragon.obj` so the high-resolution display mesh can
be written and visualized.


In [ ]:
deformed_surface = surface_embedding.deform(result.x)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
deformed_obj = OUTPUT_DIR / "dragon_static_solve_deformed.obj"
pgo.mesh.write_obj(str(deformed_obj), deformed_surface)

displacement = result.x.reshape((-1, 3))
deformed_volume_vertices = tet_data.vertices + displacement

print("rest tet bbox:    ", tet_data.bbox)
print("deformed tet bbox:", (deformed_volume_vertices.min(axis=0), deformed_volume_vertices.max(axis=0)))
print("deformed surface:", deformed_surface.num_vertices, "vertices,", deformed_surface.num_elements, "triangles")
print("wrote OBJ:", deformed_obj)

vis.plot_surface(
    [rest_surface, deformed_surface],
    titles=["dragon rest surface", "dragon static solve"],
    colors=["lightgray", "salmon"],
    show_edges=False,
    window_size=(1000, 520),
)
